# 운전 콘솔 앱 (OOP)

## 객체지향프로그래밍 OOP
- 현실세계의 모든 사건은 객체(object)와 객체사이의 상호작용인 것에 주목한다.
- 객체간의 상호작용을 구현하기 위해 의인화가 적용된다.
- 객체는 스스로 책임을 가지고 행동을 수행할 수 있다.
- 객체는 적절한 책임을 가질수 있도록 클래스 단위로 잘 분리해 작성해야 한다.
- 단일책임원칙 SRP Single Responsibility Principle 하나의 객체는 하나의 책임만 진다.
- (SOLID 객체지향 개발원칙을 참고)

## 객체간의 상호작용
- 객체는 아는 것(속성)과 할수 있는 것(메서드)으로 구성된다.
- 객체는 자기 할수 있는 것을 메서드로써 외부에 노출한다.
- 객체와 객체는 메세지를 주고받는다.
- 송신자객체는 수신자객체에 매개인자를 통해 메소드를 호출함으로써 메시지를 전송한다.
- 수신자객체는 일련의 책임을 수행후 리턴값으로 송신자객체에 응답한다.

## Driving Application

### 1. 요구사항 명세 (클라이언트)
- 운전프로그램을 만들어 주세요.
- 운전자는 시동걸기/끄기, 악셀 또는 브레이크를 밟을 수 있습니다.
- 자동차는 엔진시작/끝, 가속/감속을 할 수 있습니다.
- 자동차는 처음에 대기상태 있어야 합니다. (시동이 꺼진 상태)
- 자동차는 운전자에 의해 시동이 걸리고, 이미 시동이 걸려있다면, 또 시동을 걸수는 없습니다.
- 운전자가 악셀을 밟으면, 자동차는 10km/h씩 가속할 수 있습니다.
- 최대속도는 200km/h 입니다.
- 운전자가 브레이크를 밟으면, 자동차는 10km/h씩 감속할 수 있습니다.
- 운전자가 시동을 끄면, 자동차는 더이상 움직일 수 없습니다. (시동이 꺼진 상태)
- 자동차가 달리는 동안에는 시동을 끌수 없습니다.

### 2. 객체 도출
- 운전자
- 자동차
- 프로그램메뉴 (사용자 보게될 메뉴, 입력폼, 결과출력 담당할 UI객체)

### 3. 구현 포인트

요구사항은 단순히 속도만 올리고 내리는 것이 아니라, 현재 자동차의 상태를 확인하면서 동작을 제한해야 한다.

- 시동이 꺼져 있으면 가속/감속 불가
- 이미 시동이 걸려 있으면 다시 시동 걸기 불가
- 속도가 0보다 크면 시동 끄기 불가
- 최대 속도를 초과하지 않도록 제한

노트북에서는 `input()`으로 실행을 멈추지 않도록 샘플 메뉴 번호를 전달해 실행한다.


In [6]:
from enum import Enum


# 자동차의 시동 상태를 Enum 상수로 표현
# - 문자열 'OFF', 'ON'을 직접 비교하는 대신 의미 있는 상수로 다룬다
class CarState(Enum):
    OFF = '시동 꺼짐'   # 대기상태 (처음 상태)
    ON = '시동 켜짐'


# 자동차 객체
# - 아는 것(속성): 시동 상태, 현재 속도
# - 할 수 있는 것(메서드): 엔진 시작/끄기, 가속/감속
# - 캡슐화: 속성은 __로 숨기고, 메서드를 통해서만 상태를 바꾼다
# - 각 메서드는 처리 결과 메시지를 문자열로 return 한다 (운전자에게 응답)
class Car:
    # 클래스 속성: 모든 자동차가 공유하는 규칙
    MAX_SPEED = 200    # 최대 속도 (km/h)
    SPEED_STEP = 10    # 한 번에 가속/감속하는 단위 (km/h)

    def __init__(self):
        self.__state = CarState.OFF   # 처음에는 대기상태(시동 꺼짐)
        self.__speed = 0              # 처음 속도는 0

    # 엔진 시작
    def start_engine(self):
        # 이미 시동이 걸려 있으면 또 걸 수 없다
        if self.__state == CarState.ON:
            return '이미 시동이 걸려 있습니다.'
        self.__state = CarState.ON
        return '시동을 걸었습니다. (부릉부릉)'

    # 엔진 끄기
    def stop_engine(self):
        if self.__state == CarState.OFF:
            return '이미 시동이 꺼져 있습니다.'
        # 달리는 동안(속도가 0보다 큼)에는 시동을 끌 수 없다
        if self.__speed > 0:
            return f'달리는 중에는 시동을 끌 수 없습니다. (현재 {self.__speed}km/h)'
        self.__state = CarState.OFF
        return '시동을 껐습니다.'

    # 가속
    def increase_speed(self):
        # 시동이 꺼져 있으면 가속 불가
        if self.__state == CarState.OFF:
            return '시동이 꺼져 있어 가속할 수 없습니다.'
        # 최대 속도를 초과하지 않도록 제한
        if self.__speed >= self.MAX_SPEED:
            return f'이미 최고 속도입니다. ({self.MAX_SPEED}km/h)'
        self.__speed += self.SPEED_STEP
        return f'가속! 현재 속도: {self.__speed}km/h'

    # 감속
    def decrease_speed(self):
        # 시동이 꺼져 있으면 감속 불가
        if self.__state == CarState.OFF:
            return '시동이 꺼져 있어 감속할 수 없습니다.'
        # 속도가 0 미만으로 내려가지 않도록 제한
        if self.__speed == 0:
            return '이미 정지해 있습니다.'
        self.__speed -= self.SPEED_STEP
        return f'감속! 현재 속도: {self.__speed}km/h'

    def __str__(self):
        return f'Car(state={self.__state.value}, speed={self.__speed}km/h)'

In [7]:
# 운전자 객체
# - 자동차를 직접 조작하지 않고, 자동차에게 메시지(메서드 호출)를 보낸다
# - Driver has-a Car : 운전자는 자동차를 직접 생성해 가지고 있다
# - 자동차가 돌려준 결과 메시지를 그대로 return 해서 외부(메뉴)에 전달한다
class Driver:

    def __init__(self):
        self.__car = Car()

    def start_car(self):
        return self.__car.start_engine()

    def stop_car(self):
        return self.__car.stop_engine()

    def accelerate(self):
        return self.__car.increase_speed()

    def brake(self):
        return self.__car.decrease_speed()

In [8]:
# 프로그램 메뉴 객체 (UI 담당)
# - 사용자에게 메뉴를 보여주고, 입력을 받아 운전자에게 명령을 전달한다
# - 운전자가 돌려준 결과 메시지를 화면에 출력한다
# - 단일책임원칙(SRP): 화면 출력/입력 처리만 담당하고, 상태 규칙은 Car가 책임진다
class ConsoleMenu:
    def __init__(self, driver):
        self.__driver = driver

    # 메뉴 출력
    def show_menu(self):
        print('==================================')
        print(' 1. 시동 걸기')
        print(' 2. 시동 끄기')
        print(' 3. 악셀 밟기 (가속)')
        print(' 4. 브레이크 밟기 (감속)')
        print(' 0. 종료')
        print('==================================')

    # 선택한 메뉴 번호에 따라 운전자에게 명령 전달 후, 결과 메시지 출력
    def select(self, choice):
        if choice == '1':
            print(self.__driver.start_car())
        elif choice == '2':
            print(self.__driver.stop_car())
        elif choice == '3':
            print(self.__driver.accelerate())
        elif choice == '4':
            print(self.__driver.brake())
        elif choice == '0':
            print('프로그램을 종료합니다.')
        else:
            print('잘못된 입력입니다. 메뉴 번호를 다시 확인하세요.')

    # 실제 콘솔에서 input()으로 반복 실행하는 메서드
    # (노트북에서는 input()이 실행을 멈추므로 아래 데모 셀로 대체 확인)
    def run(self):
        while True:
            self.show_menu()
            choice = input('메뉴 선택: ')
            self.select(choice)
            if choice == '0':
                break

In [ ]:
# 객체 생성 및 상호작용
# 운전자가 자동차를 직접 생성하므로, 운전자와 메뉴만 만들면 된다
driver = Driver()
menu = ConsoleMenu(driver)

menu.run()

 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
시동이 꺼져 있어 가속할 수 없습니다.
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
이미 시동이 꺼져 있습니다.
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
시동을 걸었습니다. (부릉부릉)
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
이미 시동이 걸려 있습니다.
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
이미 시동이 걸려 있습니다.
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
가속! 현재 속도: 10km/h
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
달리는 중에는 시동을 끌 수 없습니다. (현재 10km/h)
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
가속! 현재 속도: 20km/h
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
감속! 현재 속도: 10km/h
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
감속! 현재 속도: 0km/h
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
이미 정지해 있습니다.
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
시동을 껐습니다.
 1. 시동 걸기
 2. 시동 끄기
 3. 악셀 밟기 (가속)
 4. 브레이크 밟기 (감속)
 0. 종료
프로그램을 종료합니다.
